# Análises de Negócio

Notebook de análises construído sobre a camada **gold** do modelo dimensional de atendimento V-Credit. Contempla dores de negócio identificadas:

1. **Gerente Financeiro** — gasto crescente que não acompanha a receita; necessidade de cortar despesa
2. **Coordenadora de Customer Success** — tempo médio de atendimento subindo, taxa de resolução caindo (especialmente nível 2), clientes desistindo antes do atendimento
3. **Diretor de Tecnologia** — chatbot não resolve a maioria das solicitações, inconsistências entre sistemas, necessidade de integração e governança de dados

In [0]:
%sql
USE CATALOG vcredit_medallion;

In [0]:
%sql
-- ============================================================
-- VISÃO GERAL — KPIs TOTAIS
-- ============================================================
SELECT
  COUNT(*)                                                    AS total_chamados,
  ROUND(SUM(custo), 2)                                        AS custo_total,
  ROUND(AVG(custo), 2)                                        AS custo_medio_por_chamado,
  ROUND(100 * COUNT_IF(FK_status_chamado = 1) / COUNT(*), 1)  AS pct_resolvidos,
  ROUND(100 * COUNT_IF(FK_status_chamado = 2) / COUNT(*), 1)  AS pct_nao_resolvidos,
  ROUND(100 * COUNT_IF(FK_status_chamado = 3) / COUNT(*), 1)  AS pct_abandonados,
  ROUND(100 * COUNT_IF(nota_avaliacao IS NOT NULL) / COUNT(*), 1) AS pct_com_avaliacao,
  ROUND(AVG(nota_avaliacao), 2)                               AS nota_media,
  ROUND(100 * COUNT_IF(nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5,
  ROUND(AVG(tempo_espera), 1)                                 AS tempo_medio_espera_seg,
  ROUND(AVG(tempo_atendimento), 1)                            AS tempo_medio_atendimento_seg
FROM gold.fato_chamado;

## 1. Gerente Financeiro — Contenção de Custo

> "O gasto com atendimento passou de R$ 1,4 milhão nos últimos dezenove meses e cresce todo mês. O crescimento não acompanha a receita. Precisamos cortar essa despesa."

**Análises elaboradas:**
- 1.1 Evolução mensal do custo total
- 1.2 Custo por canal de atendimento
- 1.3 Custo por motivo do chamado
- 1.4 Custo de ineficiência (não resolvidos + abandonados)
- 1.5 Custo e eficiência por nível de atendente

In [0]:
%sql
-- 1.1 — Evolução mensal do custo total de atendimento
-- Mostra se o custo cresce mês a mês; separa custo de resolvidos vs não resolvidos
SELECT
  d.ano,
  d.mes,
  COUNT(*)                         AS chamados,
  ROUND(SUM(f.custo), 2)           AS custo_total,
  ROUND(AVG(f.custo), 2)           AS custo_medio,
  ROUND(SUM(CASE WHEN f.FK_status_chamado = 1 THEN f.custo ELSE 0 END), 2)  AS custo_resolvidos,
  ROUND(SUM(CASE WHEN f.FK_status_chamado <> 1 THEN f.custo ELSE 0 END), 2) AS custo_nao_resolvidos
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, d.mes
ORDER BY d.ano, d.mes;

In [0]:
%sql
-- 1.2 — Custo por canal de atendimento
-- Onde o dinheiro está concentrado?
SELECT
  c.nome                        AS canal,
  c.tipo                        AS tipo_canal,
  COUNT(*)                      AS chamados,
  ROUND(SUM(f.custo), 2)        AS custo_total,
  ROUND(AVG(f.custo), 2)        AS custo_medio,
  ROUND(100 * SUM(f.custo) / (SELECT SUM(custo) FROM gold.fato_chamado), 1) AS pct_custo_total
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE c.SK_canal <> -1
GROUP BY c.nome, c.tipo
ORDER BY custo_total DESC;

In [0]:
%sql
-- 1.3 — Custo por motivo do chamado (top 10 por custo total)
-- Quais problemas custam mais para a empresa?
SELECT
  m.nome                        AS motivo,
  m.categoria,
  m.criticidade,
  COUNT(*)                      AS chamados,
  ROUND(SUM(f.custo), 2)        AS custo_total,
  ROUND(AVG(f.custo), 2)        AS custo_medio,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao
FROM gold.fato_chamado f
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
WHERE m.SK_motivo <> -1
GROUP BY m.nome, m.categoria, m.criticidade
ORDER BY custo_total DESC
LIMIT 10;

In [0]:
%sql
-- 1.4 — Custo de ineficiência: chamados não resolvidos e abandonados
-- Quanto dinheiro é gasto em chamados que não geram valor?
SELECT
  CASE
    WHEN f.FK_status_chamado = 1 THEN 'Resolvido'
    WHEN f.FK_status_chamado = 2 THEN 'Não resolvido (atendido)'
    WHEN f.FK_status_chamado = 3 THEN 'Abandonado (na espera)'
  END                          AS status_chamado,
  COUNT(*)                     AS chamados,
  ROUND(SUM(f.custo), 2)       AS custo_total,
  ROUND(100 * SUM(f.custo) / (SELECT SUM(custo) FROM gold.fato_chamado), 1) AS pct_custo_total
FROM gold.fato_chamado f
GROUP BY 1
ORDER BY custo_total DESC;

In [0]:
%sql
-- 1.5 — Custo e eficiência por nível de atendente
-- Nível 1 = Atendimento Inicial; Nível 2 = Atendimento Especializado
SELECT
  a.nivel_atendimento,
  COUNT(*)                              AS chamados,
  ROUND(SUM(f.custo), 2)               AS custo_total,
  ROUND(AVG(f.custo), 2)               AS custo_medio,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(AVG(f.tempo_atendimento), 1)    AS tempo_medio_atend_seg
FROM gold.fato_chamado f
JOIN gold.dim_atendente a ON f.FK_atendente = a.SK_atendente
WHERE a.SK_atendente <> -1
GROUP BY a.nivel_atendimento
ORDER BY a.nivel_atendimento;

## 2. Coordenadora de Customer Success — Qualidade do Atendimento

> "O tempo médio de atendimento subiu e a taxa de resolução caiu, principalmente no nível 2. Muitos clientes desistem antes de serem atendidos."

**Análises:**
- 2.1 Evolução mensal: tempo médio x taxa de resolução
- 2.2 Taxa de resolução e abandono por canal
- 2.3 Taxa de abandono vs tempo de espera
- 2.4 Ranking de atendentes
- 2.5 CSAT por motivo e canal

In [0]:
%sql
-- 2.1 — Evolução mensal: tempo médio de atendimento x taxa de resolução
-- Confirma se o tempo sobe e a resolução cai ao longo do tempo
SELECT
  d.ano,
  d.mes,
  COUNT(*)                                                  AS chamados,
  ROUND(AVG(f.tempo_atendimento), 1)                         AS tempo_medio_atend_seg,
  ROUND(AVG(f.tempo_espera), 1)                             AS tempo_medio_espera_seg,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 3) / COUNT(*), 1) AS pct_abandono
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, d.mes
ORDER BY d.ano, d.mes;

In [0]:
%sql
-- 2.2 — Taxa de resolução e abandono por canal de atendimento
-- Onde a resolução é mais crítica (especialmente nível 2 / Atendimento Especializado)?
SELECT
  c.nome                       AS canal,
  c.tipo                       AS tipo_canal,
  c.nivel_etapa,
  COUNT(*)                     AS chamados,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 2) / COUNT(*), 1) AS pct_nao_resolvido,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 3) / COUNT(*), 1) AS pct_abandono,
  ROUND(AVG(f.tempo_atendimento), 1)  AS tempo_medio_atend_seg,
  ROUND(AVG(f.tempo_espera), 1)       AS tempo_medio_espera_seg
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE c.SK_canal <> -1
GROUP BY c.nome, c.tipo, c.nivel_etapa
ORDER BY c.nivel_etapa;

In [0]:
%sql
-- 2.3 — Taxa de abandono e relação com tempo de espera (canais humanos)
-- Clientes desistem antes de serem atendidos: qual é o impacto do tempo de espera?
SELECT
  CASE
    WHEN f.tempo_espera = 0 THEN '0 min (sem espera)'
    WHEN f.tempo_espera <= 60 THEN 'até 1 min'
    WHEN f.tempo_espera <= 180 THEN '1 a 3 min'
    WHEN f.tempo_espera <= 300 THEN '3 a 5 min'
    WHEN f.tempo_espera <= 600 THEN '5 a 10 min'
    ELSE 'mais de 10 min'
  END                        AS faixa_espera,
  COUNT(*)                   AS chamados,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 3) / COUNT(*), 1) AS pct_abandono,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE c.tipo = 'Humano'
GROUP BY 1
ORDER BY MIN(f.tempo_espera);

In [0]:
%sql
-- 2.4 — Ranking de atendentes: tempo médio, resolução e CSAT
-- Quem performa melhor/pior para orientar capacitação
SELECT
  a.nome,
  a.nivel_atendimento,
  COUNT(*)                                                  AS chamados,
  ROUND(AVG(f.tempo_atendimento), 1)                       AS tempo_medio_atend_seg,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(AVG(f.nota_avaliacao), 2)                           AS nota_media,
  ROUND(100 * COUNT_IF(f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5,
  COUNT_IF(f.nota_avaliacao IS NOT NULL)                    AS avaliacoes
FROM gold.fato_chamado f
JOIN gold.dim_atendente a ON f.FK_atendente = a.SK_atendente
WHERE a.SK_atendente <> -1
GROUP BY a.nome, a.nivel_atendimento
ORDER BY pct_resolucao DESC, tempo_medio_atend_seg ASC;

In [0]:
%sql
-- 2.5 — CSAT (% notas 4–5) por motivo e canal (menores primeiro)
-- Onde a satisfação é menor e precisa de atenção
SELECT
  m.categoria,
  m.nome                        AS motivo,
  c.nome                        AS canal,
  COUNT(*)                      AS chamados,
  COUNT_IF(f.nota_avaliacao IS NOT NULL) AS com_avaliacao,
  ROUND(AVG(f.nota_avaliacao), 2) AS nota_media,
  ROUND(100 * COUNT_IF(f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5
FROM gold.fato_chamado f
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE m.SK_motivo <> -1 AND c.SK_canal <> -1
GROUP BY m.categoria, m.nome, c.nome
HAVING COUNT_IF(f.nota_avaliacao IS NOT NULL) > 10
ORDER BY csat_pct_4_5 ASC
LIMIT 15;

## 3. Diretor de Tecnologia — Chatbot e Integração de Dados

> "O chatbot não resolve a maior parte das solicitações. Há inconsistências entre os registros de atendimento, custos e notas de satisfação. Precisamos estruturar melhor a integração e a segurança."

**Análises:**
- 3.1 Efetividade do chatbot vs outros canais
- 3.2 Chatbot: taxa de resolução por motivo
- 3.3 Perfil do cliente por canal (faixa etária)
- 3.4 Cobertura e consistência de dados entre sistemas

In [0]:
%sql
-- 3.1 — Efetividade do chatbot vs outros canais
-- O chatbot resolve a maioria das solicitações? Como compara com URA e humanos?
SELECT
  c.nome                        AS canal,
  c.tipo                        AS tipo_canal,
  COUNT(*)                      AS chamados,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 3) / COUNT(*), 1) AS pct_abandono,
  ROUND(AVG(f.tempo_atendimento), 1)  AS tempo_medio_atend_seg,
  ROUND(SUM(f.custo), 2)              AS custo_total
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE c.SK_canal <> -1
GROUP BY c.nome, c.tipo, c.nivel_etapa
ORDER BY c.nivel_etapa;

In [0]:
%sql
-- 3.2 — Chatbot: taxa de resolução por motivo
-- Quais tipos de solicitação o chatbot consegue resolver e quais não?
SELECT
  m.nome                        AS motivo,
  m.categoria,
  COUNT(*)                      AS chamados,
  ROUND(100 * COUNT_IF(f.FK_status_chamado = 1) / COUNT(*), 1) AS pct_resolucao_chatbot
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
WHERE c.nome = 'Chatbot' AND m.SK_motivo <> -1
GROUP BY m.nome, m.categoria
ORDER BY pct_resolucao_chatbot ASC;

In [0]:
%sql
-- 3.3 — Perfil do cliente por canal (faixa etária)
-- O Diretor de TI levanta que "analfabetos digitais" não usam bem o chatbot
-- A idade dos clientes do chatbot difere dos que usam atendimento humano?
SELECT
  c.nome                        AS canal,
  cl.faixa_idade,
  COUNT(*)                       AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY c.nome), 1) AS pct_no_canal
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
JOIN gold.dim_cliente cl ON f.FK_cliente = cl.SK_cliente
WHERE c.SK_canal <> -1 AND cl.SK_cliente <> -1 AND cl.faixa_idade <> 'Não Informado'
GROUP BY c.nome, cl.faixa_idade
ORDER BY c.nome,
  CASE cl.faixa_idade
    WHEN '18-24' THEN 1 WHEN '25-34' THEN 2 WHEN '35-44' THEN 3
    WHEN '45-54' THEN 4 WHEN '55-64' THEN 5 WHEN '65-74' THEN 6
    WHEN '75-84' THEN 7 WHEN '85+' THEN 8
  END;

In [0]:
%sql
-- 3.4 — Cobertura e consistência de dados entre sistemas
-- % de chamados sem custo registrado, sem nota de satisfação, e FKs apontando para "Não Informado"
-- Quantifica o tamanho das inconsistências entre os sistemas de atendimento, custos e satisfação
SELECT
  ROUND(100 * COUNT_IF(custo = 0) / COUNT(*), 1)                    AS pct_sem_custo,
  ROUND(100 * COUNT_IF(nota_avaliacao IS NULL) / COUNT(*), 1)       AS pct_sem_nota,
  ROUND(100 * COUNT_IF(FK_cliente = -1) / COUNT(*), 1)              AS pct_cliente_nao_identificado,
  ROUND(100 * COUNT_IF(FK_atendente = -1) / COUNT(*), 1)           AS pct_atendente_nao_identificado,
  ROUND(100 * COUNT_IF(FK_data_inicio_atendimento = -1) / COUNT(*), 1) AS pct_sem_data_inicio,
  COUNT(*)                                                          AS total_chamados
FROM gold.fato_chamado;